# 03 - Moire-engineered 2D hopfion array

Use a triangular moire potential as a spatially periodic anisotropy modulation to pin a 2D array of hopfions. Initialize with the analytic ansatz at each moire site, then relax under combined exchange + DMI + (spatially varying) anisotropy. With the stability-window parameters from notebook 01 plus a moire-modulated K, all seven hopfions survive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from hopfion.grid import Grid
from hopfion.topology import hopf_index
from hopfion.energy import EnergyParams, total_energy
from hopfion.lattice import triangular_sites_2d, array_hopfion
from hopfion.moire import MoirePotential
from hopfion.llg import relax
from hopfion.viz import slice_quiver

In [ ]:
# 96x96x32 grid, dx=0.3. Spacing a_moire=8 places 7 hopfions (1 center + 6 first ring)
# with center-to-center distance ~5.3 R, well outside the hopfion interaction radius.
g = Grid(96, 96, 32, 0.3, 0.3, 0.3, 'periodic')
a_moire = 8.0
sites = triangular_sites_2d(a=a_moire, n_rings=1)
print(f'{len(sites)} hopfion sites in box {g.nx*g.dx}^2 x {g.nz*g.dz}')

In [ ]:
# Moire-engineered K(r) = K0 + V0 * sum_i cos(k_i . r) over the three star vectors.
moire = MoirePotential(K0=0.7, V0=0.3, a_moire=a_moire, lattice='triangular')
K_field = moire.Ku_field(g)
plt.imshow(np.asarray(K_field)[..., g.nz//2].T, origin='lower',
           extent=[-g.nx*g.dx/2, g.nx*g.dx/2, -g.ny*g.dy/2, g.ny*g.dy/2], cmap='magma')
plt.colorbar(label='K_u(r)')
plt.scatter([s[0] for s in sites], [s[1] for s in sites], s=60, c='cyan', edgecolors='white')
plt.title('Triangular moire anisotropy + array sites'); plt.xlabel('x'); plt.ylabel('y');

In [ ]:
m = array_hopfion(g, sites, R=1.5)
Q_init = hopf_index(m, g)
print(f'Initial total Q_H = {Q_init:+.3f} (expected ~{len(sites)})')

In [ ]:
# Same micromagnetic params as notebook 01, with K replaced by the spatial moire field.
ep = EnergyParams(A_ex=1.0, D=1.5, Ku_field=K_field, easy_axis=(0,0,1), H_ext=(0,0,0.0))
print(f'Initial E = {total_energy(m, g, ep):.4f}')
m_relaxed = relax(m, g, ep, n_steps=400, dt=0.002)
print(f'Final   E = {total_energy(m_relaxed, g, ep):.4f}')
Q_final = hopf_index(m_relaxed, g)
print(f'Final   Q_H total = {Q_final:+.3f} (expected ~{len(sites)} if all hopfions survive)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
slice_quiver(m, g, plane='xy', stride=3, ax=axes[0]); axes[0].set_title('initial array, xy slice')
slice_quiver(m_relaxed, g, plane='xy', stride=3, ax=axes[1]); axes[1].set_title('after 400 relax steps')
for a in axes:
    a.scatter([s[0] for s in sites], [s[1] for s in sites], s=30, c='black', marker='+')